NAME: DAKSH YADAV
SRN: PES2UG23CS926

INSTALL

In [8]:
!pip install -q crewai groq deepeval faiss-cpu sentence-transformers \
langchain langchain-community langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-exporter-otlp-proto-http>=1.36.0, but you have opentelemetry-exporter-otlp-proto-http 1.34.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.34.1 which is incompatible.
google-adk 1.2

IMPORTS

In [28]:
from crewai import Agent, Task, Crew

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from groq import Groq
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

KNOWLEDGE BASE

In [10]:
text = """
Large Language Models (LLMs) are advanced artificial intelligence systems that are trained on vast amounts of text data to understand and generate human-like language.

They are based on transformer architectures which use self-attention mechanisms to understand context.

LLMs are trained on datasets such as books, articles, and websites. This allows them to learn grammar, patterns, and factual knowledge.

Applications include chatbots, summarization, translation, and question answering.

However, LLMs can produce hallucinations and biased outputs. They also require large computational resources.

Fine-tuning helps adapt LLMs to specific domains.

Prompt engineering helps guide outputs effectively.

Ethical concerns include bias, privacy, and misuse.

LLMs are rapidly evolving and are a key area of modern AI research.
"""

CHUNKING + FAISS

In [11]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_text(text)

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_texts(chunks, embedding)

/tmp/ipykernel_4764/2970296632.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.war

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

GROQ SETUP

In [47]:
import os
os.environ["GROQ_API_KEY"] = "your_actual_key"

TOOL 1 (RETRIEVER TOOL)

In [13]:
def retrieve_tool(query: str) -> str:
    docs = vectorstore.similarity_search(query, k=3)
    return "\n".join([doc.page_content for doc in docs])

TOOL 2 (EVALUATOR TOOL)

In [34]:
def evaluate_tool(question: str, answer: str, context: str) -> str:
    prompt = f"""
    Evaluate the answer based on:

    1. Faithfulness (Is answer grounded in context?)
    2. Relevancy (Does it answer the question?)

    Give scores between 0 and 1.

    Context:
    {context}

    Question:
    {question}

    Answer:
    {answer}

    Output format:
    Faithfulness: X
    Relevancy: X
    Verdict: PASS or FAIL
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

TOOL 3 (LLM GENERATOR)

In [15]:
def llm_generate(prompt: str) -> str:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

AGENT 1 (RAG AGENT)

In [16]:
rag_agent = Agent(
    role="RAG Answer Generator",
    goal="Answer user questions using retrieved context",
    backstory="You are an expert at answering questions strictly using retrieved knowledge.",
    verbose=True,
    allow_delegation=False
)

AGENT 2 (EVALUATOR)

In [17]:
evaluator_agent = Agent(
    role="Answer Evaluator",
    goal="Evaluate answers using faithfulness and relevancy",
    backstory="You are a strict evaluator that checks correctness and relevance.",
    verbose=True,
    allow_delegation=False
)

AGENT 3 (REVISOR)


In [18]:
revisor_agent = Agent(
    role="Answer Revisor",
    goal="Improve answers based on feedback",
    backstory="You fix answers using evaluator feedback and ensure grounding in context.",
    verbose=True,
    allow_delegation=False
)

TASK 1 (RAG TASK)

In [19]:
def rag_task_fn(question):
    context = retrieve_tool(question)

    prompt = f"""
    Use ONLY this context to answer.

    Context:
    {context}

    Question:
    {question}
    """

    answer = llm_generate(prompt)

    return {
        "question": question,
        "answer": answer,
        "context": context
    }

TASK 2 (EVALUATION TASK)

In [20]:
def eval_task_fn(data):
    return evaluate_tool(
        data["question"],
        data["answer"],
        data["context"]
    )

TASK 3 (REVISION TASK)

In [21]:
def revise_task_fn(data, evaluation):
    prompt = f"""
    Improve the answer.

    Question: {data['question']}

    Answer: {data['answer']}

    Context: {data['context']}

    Feedback: {evaluation}

    Return improved answer grounded in context.
    """

    return llm_generate(prompt)

PIPELINE FUNCTION

In [40]:
def run_pipeline(question):
    rag_output = rag_task_fn(question)

    evaluation = eval_task_fn(rag_output)


    if "FAIL" in evaluation and "Relevancy: 0.0" in evaluation:
        return {
            "question": question,
            "initial_eval": evaluation,
            "final_eval": evaluation,
            "final_answer": "The answer is not present in the knowledge base."
        }


    if "FAIL" in evaluation:
        revised_answer = revise_task_fn(rag_output, evaluation)

        final_eval = evaluate_tool(
            rag_output["question"],
            revised_answer,
            rag_output["context"]
        )

        return {
            "question": question,
            "initial_eval": evaluation,
            "final_eval": final_eval,
            "final_answer": revised_answer
        }

    return {
        "question": question,
        "initial_eval": evaluation,
        "final_eval": evaluation,
        "final_answer": rag_output["answer"]
    }

QUESTIONS

In [41]:
questions = [
    "What are large language models?",
    "How do LLMs work?",
    "What is fine-tuning?",
    "What are applications of LLMs?",
    "Why are LLMs important?",
    "What is quantum computing?",
    "Who is the president of USA?"
]

OUTPUT

In [45]:
def extract_scores(eval_text):
    lines = eval_text.strip().split("\n")
    f = float(lines[0].split(":")[1].strip())
    r = float(lines[1].split(":")[1].strip())
    v = lines[2].split(":")[1].strip()
    return f, r, v

rows = []

for r in results:
    f1, r1, v1 = extract_scores(r["initial_eval"])
    f2, r2, v2 = extract_scores(r["final_eval"])

    rows.append({
        "Question": r["question"],
        "Init Faithfulness": f1,
        "Init Relevancy": r1,
        "Init Verdict": v1,
        "Final Faithfulness": f2,
        "Final Relevancy": r2,
        "Final Verdict": v2,
        "Final Answer": r["final_answer"]
    })

import pandas as pd
df = pd.DataFrame(rows)
df

,Question,Init Faithfulness,Init Relevancy,Init Verdict,Final Faithfulness,Final Relevancy,Final Verdict,Final Answer
0,What are large language models?,1.0,1.0,PASS,1.0,1.0,PASS,Large Language Models (LLMs) are advanced arti...
1,How do LLMs work?,1.0,1.0,PASS,1.0,1.0,PASS,LLMs work by being trained on datasets such as...
2,What is fine-tuning?,1.0,1.0,PASS,1.0,1.0,PASS,Fine-tuning helps adapt LLMs to specific domains.
3,What are applications of LLMs?,1.0,1.0,PASS,1.0,1.0,PASS,The applications of LLMs include:\n\n1. Chatbo...
4,Why are LLMs important?,0.9,1.0,PASS,0.9,1.0,PASS,LLMs are important because they are a key area...
5,What is quantum computing?,1.0,0.0,FAIL,1.0,0.0,FAIL,There is no information about quantum computin...
6,Who is the president of USA?,1.0,0.0,FAIL,1.0,0.0,FAIL,The provided context does not contain informat...


## Part 6: Reflection

1) The system performed well on in-domain questions but failed on adversarial or out-of-scope queries. Questions such as “What is quantum computing?” and “Who is the president of USA?” caused failures because the knowledge base did not contain relevant information. In these cases, the retriever still returned some context, but it was unrelated to the query. As a result, the generated answer remained faithful to the retrieved context (high faithfulness score) but failed in relevancy (low relevancy score). This highlights a key limitation of RAG systems when handling queries outside the knowledge base.

2) The revision step was effective only for improving clarity or structure when the initial answer was partially incorrect. However, it did not significantly improve scores for adversarial questions because the underlying issue was the lack of relevant context, not answer quality. Once relevancy is zero, revision alone cannot fix the response.

3) To improve reliability, I would enhance the retrieval mechanism by using hybrid search (combining semantic and keyword-based retrieval) and increasing the size and diversity of the knowledge base. Additionally, introducing a confidence threshold or relevancy check before answer generation could prevent the system from producing misleading outputs.

4) For monitoring, TruLens can be integrated to track metrics such as faithfulness, relevancy, and user feedback over time. This would allow continuous evaluation, identification of failure patterns, and iterative improvement of the system in real-world usage.